## LIBRARIES

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

## WIDGETS

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("storageName", "saccexplorer")
dbutils.widgets.text("containerName", "golden")

dbutils.widgets.text("catalogName", "unit_catalog_explorer")
dbutils.widgets.text("schema_source", "uc_silver")
dbutils.widgets.text("schema_sink", "uc_golden")


## CONSTANTS

In [0]:
storage = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("containerName")

catalog         = dbutils.widgets.get("catalogName")
schema_source   = dbutils.widgets.get("schema_source")
schema_sink     = dbutils.widgets.get("schema_sink")


table_ranking    = "kpi_ranking_empresarial"
table_geografica = "kpi_ventas_geograficas"
table_sectorial    = "kpi_desempeno_sectorial"


## PATHS

In [0]:
path_base_golden = f"abfss://{container}@{storage}.dfs.core.windows.net/{schema_sink}"

path_fct_ranking = f"{path_base_golden}/{table_ranking}"
path_kpi_ventas  = f"{path_base_golden}/{table_geografica}"
path_kpi_sector  = f"{path_base_golden}/{table_sectorial}"

## READ AND CACHE

In [0]:
df_ranking_final = spark.table(f"{catalog}.{schema_source}.ranking_enriquecido")
df_kpi_provincia = spark.table(f"{catalog}.{schema_source}.ranking_por_provincia")
df_kpi_sector    = spark.table(f"{catalog}.{schema_source}.ranking_por_sector")

## SAVE

In [0]:
# 1. Tabla Maestra (Fact Table)
df_ranking_final.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_fct_ranking) \
    .saveAsTable(f"{catalog}.{schema_sink}.{table_ranking}")

# 2. KPI Ventas Geográficas
df_kpi_provincia.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_kpi_ventas) \
    .saveAsTable(f"{catalog}.{schema_sink}.{table_geografica}")

# 3. KPI Desempeño Sectorial
df_kpi_sector.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_kpi_sector) \
    .saveAsTable(f"{catalog}.{schema_sink}.{table_sectorial}")

print(f"Carga a Capa Golden finalizada. Ubicación física: {path_base_golden}")

## DEFINITIONS

In [0]:
%skip
import pandas as pd

# 1. Definir los datos del glosario
glosario_datos = [
    # Datos de Compañía
    ("expediente", "Número identificador único de la compañía otorgado por la Superintendencia de Compañías, Valores y Seguros (SCVS)."),
    ("ruc", "Registro Único de Contribuyentes; número identificador tributario de la compañía otorgado por el Servicio de Rentas Internas (SRI)."),
    ("nombre", "Nombre o razón social de la compañía."),
    ("tipo", "Clasificación jurídica o forma legal de la compañía."),
    ("pro_codigo", "Código numérico asignado a la provincia del domicilio legal."),
    ("provincia", "Nombre de la provincia donde está domiciliada la compañía."),
    
    # Datos de Segmento
    ("id_segmento", "Número identificador del segmento al que pertenece una compañía."),
    ("segmento", "Descripción del segmento al que pertenece una compañía."),

    # Datos de CIIU
    ("ciiu", "Código de Clasificación Industrial Internacional Uniforme de la actividad económica (todos los niveles)."),
    ("descripcion", "Descripción del código ciiu."),
    
    # Datos Financieros e Indicadores
    ("anio", "Año fiscal en que se presenta el estado financiero."),
    ("posicion_general", "Número de la posición de la compañía."),
    ("cia_imvalores", "Marca si la compañía pertenece al sector de mercado de valores: 1 (Sí); 0 (No)."),
    ("id_estado_financiero", "Número identificador del estado financiero."),
    ("ingresos_ventas", "Sumatoria de los ingresos por ventas de acuerdo a lo registrado en el estado financiero."),
    ("activos", "Sumatoria de los activos de acuerdo a lo registrado en el estado financiero."),
    ("patrimonio", "Sumatoria del patrimonio de acuerdo a lo registrado en el estado financiero."),
    ("utilidad_an_imp", "Utilidad antes de impuestos; cifra resultante tras deducir costos y gastos (excepto impuestos) del total de ingresos."),
    ("impuesto_renta", "Valor aplicado sobre las ganancias obtenidas en el año fiscal registrado."),
    ("n_empleados", "Número de empleados registrado en el estado financiero."),
    ("ingresos_totales", "Sumatoria de todos los ingresos recibidos en el año fiscal registrado."),
    ("utilidad_ejercicio", "Ganancia obtenida por ventas de productos o servicios tras descontar costos de producción."),
    ("utilidad_neta", "Ganancia final tras descontar costos de producción, distribución, logística, gastos operativos, impuestos y obligaciones."),
    ("cod_segmento", "Número identificador del tipo de segmento al que pertenece la compañía."),
    ("ciiu_n1", "Código de Clasificación Industrial Internacional Uniforme a nivel 1 de la actividad económica."),
    ("ciiu_n6", "Código de Clasificación Industrial Internacional Uniforme a nivel 6 de la actividad económica."),
    ("liquidez_corriente", "Indicador de liquidez corriente calculado con los valores del estado financiero."),
    ("prueba_acida", "Indicador de prueba ácida calculado con los valores del estado financiero."),
    ("end_activo", "Indicador de endeudamiento del activo calculado con los valores del estado financiero."),
    ("end_patrimonial", "Indicador de endeudamiento patrimonial calculado con los valores del estado financiero."),
    ("end_activo_fijo", "Indicador de endeudamiento del activo fijo calculado con los valores del estado financiero."),
    ("end_corto_plazo", "Indicador de endeudamiento a corto plazo calculado con los valores del estado financiero."),
    ("end_largo_plazo", "Indicador de endeudamiento a largo plazo calculado con los valores del estado financiero."),
    ("cobertura_interes", "Indicador de cobertura de interés calculado con los valores del estado financiero."),
    ("apalancamiento", "Indicador de apalancamiento calculado con los valores del estado financiero."),
    ("apalancamiento_financiero", "Indicador de apalancamiento financiero calculado con los valores del estado financiero."),
    ("end_patrimonial_ct", "Indicador de endeudamiento patrimonial corriente calculado con los valores del estado financiero."),
    ("end_patrimonial_nct", "Indicador de endeudamiento patrimonial no corriente calculado con los valores del estado financiero."),
    ("apalancamiento_c_l_plazo", "Indicador de apalancamiento a corto y largo plazo calculado con los valores del estado financiero."),
    ("rot_cartera", "Indicador de rotación de cartera calculado con los valores del estado financiero."),
    ("rot_activo_fijo", "Indicador de rotación de activo fijo calculado con los valores del estado financiero."),
    ("rot_ventas", "Indicador de rotación de ventas calculado con los valores del estado financiero."),
    ("per_med_cobranza", "Indicador de período medio de cobranza calculado con los valores del estado financiero."),
    ("per_med_pago", "Indicador de período medio de pago calculado con los valores del estado financiero."),
    ("impac_gasto_a_v", "Indicador de impacto de gastos de administración y ventas calculado con los valores del estado financiero."),
    ("impac_carga_finan", "Indicador de impacto de la carga financiera calculado con los valores del estado financiero."),
    ("margen_bruto", "Indicador de margen bruto calculado con los valores del estado financiero."),
    ("margen_operacional", "Indicador de margen operacional calculado con los valores del estado financiero."),
    ("rent_neta_ventas", "Indicador de rentabilidad neta de ventas calculado con los valores del estado financiero."),
    ("rent_ope_patrimonio", "Indicador de rentabilidad operacional del patrimonio calculado con los valores del estado financiero."),
    ("rent_ope_activo", "Indicador de rentabilidad operacional del activo calculado con los valores del estado financiero."),
    ("roe", "Indicador de rentabilidad financiera calculado con los valores del estado financiero."),
    ("roa", "Indicador de rendimiento de los activos calculado con los valores del estado financiero."),
    ("fortaleza_patrimonial", "Indicador de fortaleza patrimonial calculado con los valores del estado financiero."),
    ("gastos_financieros", "Sumatoria de los gastos financieros registrados en el estado financiero."),
    ("gastos_admin_ventas", "Sumatoria de los gastos administrativos y de ventas registrados en el estado financiero."),
    ("depreciaciones", "Sumatoria de los gastos por depreciaciones registrados en el estado financiero."),
    ("amortizaciones", "Sumatoria de los gastos por amortizaciones registrados en el estado financiero."),
    ("costos_ventas_prod", "Sumatoria de los costos de ventas y producción registrados en el estado financiero."),
    ("deuda_total", "Sumatoria de todas las obligaciones financieras registradas en el estado financiero."),
    ("deuda_total_c_plazo", "Sumatoria de las obligaciones financieras corrientes (corto plazo) registradas en el estado financiero."),
    ("total_gastos", "Sumatoria total de los gastos registrados en el estado financiero.")
]

# 2. Crear el DataFrame
df_glosario = spark.createDataFrame(glosario_datos, ["termino", "definicion"])

# 3. Guardar como tabla permanente en Databricks (Delta Lake)
df_glosario.write.mode("overwrite").saveAsTable("lakehouse_gtyt_dev.db_bronze.supercias_glosario")

# 4. Mostrar el resultado
display(spark.table("lakehouse_gtyt_dev.db_bronze.supercias_glosario"))